# PLaMo 3 Translate を Colab で OpenAI 互換 API に

構成: `llama-server`（plamo-3-translate-0.8b-Q4_K_M, :8000）→ `plamo_proxy`（バッチ分解＋APIキー認証＋記録, :8100）→ Cloudflare Tunnel（https）

1. セルを上から順に実行
2. ④で表示される **接続先URL / APIキー / モデル名** をアプリの「OpenAI 互換（カスタム）」に入力

0.8B なので CPU ランタイムでも動きます。リクエストと返答は `/content/requests.jsonl` に記録されます。

In [ ]:
#@title ① 設定
API_KEY = ""  #@param {type:"string"}
#@markdown 空にするとランダム生成

#@markdown ---
#@markdown ### 実行環境
USE_GPU = False  #@param {type:"boolean"}
USE_DRIVE_LLAMA = True  #@param {type:"boolean"}
DRIVE_LLAMA_DIR = "/content/drive/MyDrive/llama"  #@param {type:"string"}
#@markdown ON: Google ドライブの llama フォルダにある llama-server（llama-server-static / Linux 用 zip）を使う。無い・動かない場合は自動でダウンロード／ビルドに切替
#@markdown ON: CUDA 版 llama.cpp をビルド（GPU ランタイム必須・初回 10〜15 分）／ OFF: CPU 版をダウンロード（すぐ起動）

#@markdown ---
#@markdown ### Context and Offload
CTX_SIZE = 8192  #@param {type:"integer"}
#@markdown コンテキスト長（全スロット合計）。マイクラの lang 1行は短いので 8192 で十分
GPU_LAYERS = 99  #@param {type:"integer"}
#@markdown GPUオフロード層数（USE_GPU=OFF のときは無視。99 = 全部）

#@markdown ---
#@markdown ### Advanced
THREADS = 0  #@param {type:"integer"}
#@markdown CPUスレッド数（0 = 自動）
BATCH_SIZE = 2048  #@param {type:"integer"}
#@markdown 評価バッチサイズ (-b)
UBATCH_SIZE = 512  #@param {type:"integer"}
#@markdown Physical Batch Size (-ub)
PARALLEL = 4  #@param {type:"integer"}
#@markdown Max Concurrent（同時翻訳数。プロキシの並列数も同じ値になります）
KV_UNIFIED = True  #@param {type:"boolean"}
#@markdown Unified KV Cache（ON 推奨: 短い文字列を並列で流すマイクラ翻訳向き）
CTX_CHECKPOINTS = 32  #@param {type:"integer"}
ROPE_FREQ_BASE = 0.0  #@param {type:"number"}
#@markdown 0 = 自動
ROPE_FREQ_SCALE = 0.0  #@param {type:"number"}
#@markdown 0 = 自動
KV_OFFLOAD = True  #@param {type:"boolean"}
#@markdown KVキャッシュをGPUメモリにオフロード
MLOCK = True  #@param {type:"boolean"}
#@markdown モデルをメモリに保持
USE_MMAP = True  #@param {type:"boolean"}
SEED = -1  #@param {type:"integer"}
#@markdown -1 = ランダム（temperature 0 なので結果はほぼ固定）
FLASH_ATTN = "on"  #@param ["on", "off", "auto"]

#@markdown ---
#@markdown ### 翻訳プロキシ（マイクラ用）
MAX_TOKENS = 512  #@param {type:"integer"}
#@markdown 1エントリあたりの最大出力トークン（本・説明文が長い MOD は 1024 に）
TEMPERATURE = 0.0  #@param {type:"number"}
REQUEST_TIMEOUT = 120  #@param {type:"integer"}
#@markdown 1エントリあたりのタイムアウト秒

if PARALLEL < 1: PARALLEL = 1
if not KV_UNIFIED and CTX_SIZE // PARALLEL < 1024:
    print(f"⚠ 1スロットあたり {CTX_SIZE // PARALLEL} トークンしかありません。CTX_SIZE を増やすか KV_UNIFIED を ON に")
MODEL_URL = "https://huggingface.co/pfnet/plamo-3-translate-0.8b-GGUF/resolve/main/plamo-3-translate-0.8b-Q4_K_M.gguf?download=true"
MODEL_PATH = "/content/plamo-3-translate-0.8b-Q4_K_M.gguf"
MODEL_NAME = "plamo-3-translate"

import secrets
if not API_KEY:
    API_KEY = "sk-colab-" + secrets.token_urlsafe(24)
print("API_KEY:", API_KEY)

In [ ]:
#@title ② llama.cpp・cloudflared・モデルを取得して llama-server を起動
import json, os, glob, subprocess, time, urllib.request, zipfile, tarfile

import shutil
if USE_GPU:
    if shutil.which("nvidia-smi") is None or subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
        print("⚠ GPU が見つかりません（CPU ランタイムです）→ CPU 版で続行します")
        print("   GPU を使うには「ランタイム → ランタイムのタイプを変更」で T4 GPU を選んで再実行")
        USE_GPU = False

# CUDA ランタイムのライブラリパス（CUDA 版バイナリ用）
LIB_DIRS = [d for d in ["/usr/local/cuda/lib64", "/usr/lib64-nvidia", "/usr/local/nvidia/lib64"] if os.path.isdir(d)]

def try_binary(path, extra_dirs=()):
    """実行できるか --version で確認"""
    os.chmod(path, 0o755)
    e = dict(os.environ, LD_LIBRARY_PATH=":".join([os.path.dirname(path), *extra_dirs, *LIB_DIRS]))
    r = subprocess.run([path, "--version"], capture_output=True, text=True, env=e)
    print((r.stdout + r.stderr).strip()[-500:])
    return r.returncode == 0

LLAMA_SERVER = None
if USE_DRIVE_LLAMA:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_LLAMA_DIR):
        print("⚠ ドライブに", DRIVE_LLAMA_DIR, "がありません")
    else:
        print("ドライブ:", sorted(os.listdir(DRIVE_LLAMA_DIR)))
        os.makedirs("/content/llama_drive", exist_ok=True)
        # 1) 公式ビルドの zip / tar.gz を展開（cudart-* は CUDA ライブラリとして一緒に展開）
        for name in sorted(os.listdir(DRIVE_LLAMA_DIR)):
            src = os.path.join(DRIVE_LLAMA_DIR, name)
            if "win" in name.lower():
                print("スキップ（Windows 用）:", name); continue
            if "cuda" in name.lower() and not USE_GPU:
                print("スキップ（CUDA 版・USE_GPU=OFF）:", name); continue
            if name.endswith(".zip"):
                zipfile.ZipFile(src).extractall("/content/llama_drive")
            elif name.endswith((".tar.gz", ".tgz")):
                tarfile.open(src).extractall("/content/llama_drive")
        libdirs = {os.path.dirname(p) for p in glob.glob("/content/llama_drive/**/*.so*", recursive=True)}
        for cand in glob.glob("/content/llama_drive/**/llama-server", recursive=True):
            print("試行:", cand)
            if try_binary(cand, libdirs):
                LLAMA_SERVER = cand; LIB_DIRS[:0] = list(libdirs); break
            print("  → 実行できませんでした")
        # 2) llama-server-static（単体バイナリ。対応オプションが少ないことがある）
        if LLAMA_SERVER is None:
            for name in sorted(os.listdir(DRIVE_LLAMA_DIR)):
                if name.startswith("llama-server") and not name.endswith((".zip", ".tar.gz", ".exe")):
                    dst = "/content/llama_drive/" + name
                    shutil.copy(os.path.join(DRIVE_LLAMA_DIR, name), dst)
                    print("試行:", name)
                    if try_binary(dst):
                        LLAMA_SERVER = dst; break
                    print("  → 実行できませんでした")
    if LLAMA_SERVER:
        print("✅ ドライブの llama-server を使用:", LLAMA_SERVER)
    else:
        print("⚠ ドライブに使える Linux 用 llama-server がありません → 通常の方法で取得します")

if USE_GPU and not LLAMA_SERVER:
    # CUDA 版をソースからビルド（2回目以降はスキップ）
    LLAMA_SERVER = "/content/llama.cpp/build/bin/llama-server"
    if not os.path.exists(LLAMA_SERVER):
        cuda_bin = next((d for d in ["/usr/local/cuda/bin", *sorted(glob.glob("/usr/local/cuda-*/bin"), reverse=True)]
                         if os.path.exists(d + "/nvcc")), None)
        if cuda_bin is None:
            raise RuntimeError("nvcc（CUDA Toolkit）が見つかりません。USE_GPU を OFF にして実行してください")
        os.environ["PATH"] = cuda_bin + ":" + os.environ["PATH"]
        os.environ["CUDACXX"] = cuda_bin + "/nvcc"
        os.environ["CUDAToolkit_ROOT"] = os.path.dirname(cuda_bin)
        cap = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.split()[0].replace(".", "")
        print("CUDA:", cuda_bin, "/ compute capability:", cap)
        shutil.rmtree("/content/llama.cpp", ignore_errors=True)
        !git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
        !cd /content/llama.cpp && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={cap} -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -n 5 && cmake --build build --target llama-server -j$(nproc) 2>&1 | tail -n 3
        if not os.path.exists(LLAMA_SERVER):
            raise RuntimeError("CUDA ビルドに失敗しました（上のログを確認）。USE_GPU を OFF にすれば CPU 版で動きます")

if not LLAMA_SERVER:
    # llama.cpp の最新 Ubuntu x64 (CPU) ビルドを取得
    rel = json.load(urllib.request.urlopen("https://api.github.com/repos/ggml-org/llama.cpp/releases/latest"))
    asset = next(a for a in rel["assets"] if "bin-ubuntu-x64" in a["name"])
    print("llama.cpp:", rel["tag_name"], asset["name"])
    urllib.request.urlretrieve(asset["browser_download_url"], "/content/llama_bin")
    os.makedirs("/content/llama", exist_ok=True)
    if asset["name"].endswith(".zip"):
        zipfile.ZipFile("/content/llama_bin").extractall("/content/llama")
    else:
        tarfile.open("/content/llama_bin").extractall("/content/llama")
    LLAMA_SERVER = glob.glob("/content/llama/**/llama-server", recursive=True)[0]
os.chmod(LLAMA_SERVER, 0o755)

!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared

if not os.path.exists(MODEL_PATH):
    !wget -q --show-progress -O {MODEL_PATH} "{MODEL_URL}"

env = dict(os.environ, LD_LIBRARY_PATH=":".join([os.path.dirname(LLAMA_SERVER), *LIB_DIRS]))
args = [LLAMA_SERVER, "-m", MODEL_PATH, "--host", "127.0.0.1", "--port", "8000",
        "--alias", MODEL_NAME, "--jinja", "--no-prefill-assistant",
        "-c", str(CTX_SIZE), "-np", str(PARALLEL), "-b", str(BATCH_SIZE), "-ub", str(UBATCH_SIZE),
        "-ngl", str(GPU_LAYERS if USE_GPU else 0), "--seed", str(SEED), "-fa", FLASH_ATTN,
        "--ctx-checkpoints", str(CTX_CHECKPOINTS)]
if THREADS > 0:         args += ["-t", str(THREADS)]
if KV_UNIFIED:          args += ["--kv-unified"]
if ROPE_FREQ_BASE > 0:  args += ["--rope-freq-base", str(ROPE_FREQ_BASE)]
if ROPE_FREQ_SCALE > 0: args += ["--rope-freq-scale", str(ROPE_FREQ_SCALE)]
if not KV_OFFLOAD:      args += ["--no-kv-offload"]
if MLOCK:               args += ["--mlock"]
if not USE_MMAP:        args += ["--no-mmap"]
# このビルドが対応していないオプションは外す（例: 古い/派生ビルドの --mlock など）
import re
help_txt = subprocess.run([LLAMA_SERVER, "--help"], capture_output=True, text=True, env=env)
help_txt = help_txt.stdout + help_txt.stderr
REQUIRED = {"-m", "--host", "--port"}
kept, i = [args[0]], 1
while i < len(args):
    opt = args[i]
    has_val = i + 1 < len(args) and not args[i + 1].startswith("--") and not (args[i + 1].startswith("-") and not args[i + 1].lstrip("-").replace(".", "").isdigit())
    chunk = args[i:i + 2] if has_val else args[i:i + 1]
    if opt in REQUIRED or re.search(r"(^|[\s,])" + re.escape(opt) + r"([\s,=]|$)", help_txt, re.M):
        kept += chunk
    else:
        print("⚠ 未対応のため無視:", " ".join(chunk))
    i += len(chunk)
args = kept
print("起動:", " ".join(args[1:]))
if 'llama' in globals() and llama.poll() is None:  # 設定を変えて再実行したとき
    llama.terminate(); llama.wait()
llama = subprocess.Popen(args, env=env, stdout=open("/content/llama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        if json.load(urllib.request.urlopen("http://127.0.0.1:8000/health")).get("status") == "ok":
            break
    except Exception:
        pass
    if llama.poll() is not None:
        raise RuntimeError("llama-server が停止しました:\n" + open("/content/llama.log").read()[-3000:])
    time.sleep(1)
print("✅ llama-server 起動 (:8000)")

In [ ]:
%%writefile /content/plamo_proxy.py
"""OpenAI互換のバッチ翻訳リクエストを PLaMo 3 Translate 向けに1件ずつ変換するプロキシ。
Colab版: APIキー認証・/v1/models・リクエスト記録を追加。標準ライブラリのみ。
"""
import datetime
import json
import os
import re
import sys
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

PLAMO_URL = "http://127.0.0.1:8000/v1/chat/completions"
PLAMO_MODEL = "plamo-3-translate"
LISTEN_PORT = 8100
PARALLEL = int(os.environ.get("PARALLEL", "4"))  # llama-server の -np と合わせる
API_KEY = os.environ.get("API_KEY", "")
MAX_TOKENS = int(os.environ.get("MAX_TOKENS", "512"))
TEMPERATURE = float(os.environ.get("TEMPERATURE", "0"))
TIMEOUT = int(os.environ.get("REQUEST_TIMEOUT", "120"))
LOG_FILE = "/content/requests.jsonl"

LANG = {"zh_cn": "zh-hans", "zh_tw": "zh-hant", "zh_hk": "zh-hant"}

# printf系 / Minecraft書式コード / {0} / 改行
PH = re.compile(
    r"%(?:\d+\$)?[-#+ 0,(]*\d*(?:\.\d+)?[sdfxXcbeEgGoh%]"
    r"|§[0-9a-fk-orA-FK-OR]"
    r"|\{\d+\}"
    r"|\\n|\n"
)
FULLWIDTH = str.maketrans({"｛": "{", "｝": "}", "０": "0", "１": "1", "２": "2", "３": "3",
                           "４": "4", "５": "5", "６": "6", "７": "7", "８": "8", "９": "9"})


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, file=sys.stderr, flush=True)


def record(req, res, code):
    entry = {"time": datetime.datetime.now().isoformat(timespec="seconds"), "status": code,
             "request": req, "response": res}
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")


def number_s(src: str) -> str:
    """番号なし %s/%d を %1$s, %2$d ... に（語順入替で意味が逆になるのを防ぐ）。%% は除外。"""
    if re.search(r"%\d+\$", src):  # 番号付きと混在している場合は触らない
        return src
    n = iter(range(1, 100))
    return re.sub(r"(?<!%)%([sd])", lambda m: f"%{next(n)}${m.group(1)}", src)


def mask(src: str):
    phs = []

    def rep(m):
        phs.append(m.group(0))
        return f"{{{len(phs) - 1}}}"

    return PH.sub(rep, src), phs


def unmask(t: str, phs):
    t = t.translate(FULLWIDTH)
    t = re.sub(r"\{\s*(\d+)\s*\}", r"{\1}", t)
    # 置換記号以外の {n} が勝手に増えていないか
    if sorted(map(int, re.findall(r"\{(\d+)\}", t))) != list(range(len(phs))):
        return None
    for i, p in enumerate(phs):
        t = t.replace(f"{{{i}}}", p)
    return t


def to_lang(locale: str) -> str:
    loc = locale.lower().replace("-", "_")
    return LANG.get(loc, loc.split("_")[0])


def to_glossary(g):
    out = []
    for item in g or []:
        if isinstance(item, dict):
            s = item.get("source") or item.get("term") or item.get("src")
            t = item.get("target") or item.get("translation") or item.get("dst")
            if s and t:
                out.append({"source": s, "target": t})
    return out


def call_plamo(text, target_lang, context, glossary):
    body = {
        "model": PLAMO_MODEL,
        "messages": [{"role": "user", "content": text}],
        "chat_template_kwargs": {"target_lang": target_lang, "context": context, "glossary": glossary},
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "stop": ["<|plamo:tag|>", "<|plamo:bos|>"],
    }
    req = urllib.request.Request(PLAMO_URL, json.dumps(body).encode(), {"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=TIMEOUT) as r:
        return json.load(r)["choices"][0]["message"]["content"].strip()


def translate_entry(e, batch, glossary):
    src = e["source"]
    numbered = number_s(src)
    masked, phs = mask(numbered)
    if not masked.strip("{}0123456789 "):  # 置換文字だけの文字列は訳さない
        return {"id": e["id"], "translation": numbered}
    ctx = f"Minecraft mod '{batch.get('mod_name', '')}' UI string. key: {e.get('key', '')}"
    try:
        out = call_plamo(masked, to_lang(e.get("target_locale", "ja_jp")), ctx, glossary)
        t = unmask(out, phs)
    except Exception as ex:  # noqa: BLE001
        log(f"[ERROR] id={e['id']} {ex}")
        t = None
    if t is None:
        log(f"[FALLBACK] id={e['id']} key={e.get('key')} src={src!r}")
        t = numbered
    elif "§" in src:
        log(f"[CHECK §] id={e['id']} {src!r} -> {t!r}")
    return {"id": e["id"], "translation": t}


def wrap(req, content):
    return {
        "id": f"chatcmpl-plamo-{int(time.time() * 1000)}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.get("model", PLAMO_MODEL),
        "choices": [{"index": 0, "message": {"role": "assistant", "content": content}, "finish_reason": "stop"}],
        "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0},
    }


def handle(req: dict) -> dict:
    last = req["messages"][-1]["content"]
    try:
        batch = json.loads(last)
        entries = batch["entries"]
    except Exception:  # バッチ形式でなければ日本語へ単発翻訳（接続テスト用）
        log("[PLAIN] non-batch request")
        return wrap(req, call_plamo(str(last), "ja", "", []))
    glossary = to_glossary(batch.get("glossary"))
    with ThreadPoolExecutor(PARALLEL) as ex:
        out = list(ex.map(lambda e: translate_entry(e, batch, glossary), entries))
    return wrap(req, json.dumps({"translations": out}, ensure_ascii=False))


class H(BaseHTTPRequestHandler):
    def send_json(self, code, obj):
        data = json.dumps(obj, ensure_ascii=False).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(data)))
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()
        self.wfile.write(data)

    def authed(self):
        if API_KEY and self.headers.get("Authorization", "") != f"Bearer {API_KEY}":
            self.send_json(401, {"error": {"message": "Invalid API key", "type": "invalid_request_error"}})
            return False
        return True

    def do_OPTIONS(self):
        self.send_response(204)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Headers", "*")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        self.end_headers()

    def do_GET(self):
        if not self.authed():
            return
        if self.path.rstrip("/").endswith("/models"):
            self.send_json(200, {"object": "list", "data": [
                {"id": PLAMO_MODEL, "object": "model", "created": 0, "owned_by": "pfnet"}]})
        else:
            self.send_json(200, {"status": "ok"})

    def do_POST(self):
        if not self.authed():
            return
        req = None
        try:
            req = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
            if req.get("stream"):
                raise ValueError("stream is not supported")
            res, code = handle(req), 200
        except Exception as ex:  # noqa: BLE001
            log(f"[BAD REQUEST] {ex}")
            res, code = {"error": {"message": str(ex)}}, 400
        try:
            record(req, res, code)
        except Exception as ex:  # noqa: BLE001
            log(f"[LOG ERROR] {ex}")
        n = len(req.get("messages", [])) if isinstance(req, dict) else 0
        log(f"[REQ] {code} messages={n}")
        self.send_json(code, res)

    def log_message(self, *a):
        pass


if __name__ == "__main__":
    log(f"listening :{LISTEN_PORT} -> {PLAMO_URL}")
    ThreadingHTTPServer(("127.0.0.1", LISTEN_PORT), H).serve_forever()


In [ ]:
#@title ③ プロキシ起動 ＋ Cloudflare Tunnel で公開
import subprocess, re, time, threading, os
penv = dict(os.environ, API_KEY=API_KEY, PARALLEL=str(PARALLEL), MAX_TOKENS=str(MAX_TOKENS),
            TEMPERATURE=str(TEMPERATURE), REQUEST_TIMEOUT=str(REQUEST_TIMEOUT))
if 'proxy' in globals() and proxy.poll() is None:  # 再実行時は再起動
    proxy.terminate(); proxy.wait()
proxy = subprocess.Popen(["python", "/content/plamo_proxy.py"], env=penv,
                         stdout=open("/content/proxy.log", "w"), stderr=subprocess.STDOUT)
time.sleep(2)

tun = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8100"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
PUBLIC_URL = None
for line in tun.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        PUBLIC_URL = m.group(0); break
threading.Thread(target=lambda: [None for _ in tun.stdout], daemon=True).start()
time.sleep(5)

print("=" * 60)
print("アプリの「OpenAI 互換（カスタム）」に入力してください")
print("接続先URL:", PUBLIC_URL + "/v1")
print("APIキー  :", API_KEY)
print("モデル   :", MODEL_NAME)
print("=" * 60)

In [ ]:
#@title ④ 動作テスト（任意）
import json, urllib.request
h = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
def post(body):
    r = urllib.request.Request(PUBLIC_URL + "/v1/chat/completions", json.dumps(body).encode(), h)
    return json.load(urllib.request.urlopen(r, timeout=300))["choices"][0]["message"]["content"]

print(json.load(urllib.request.urlopen(urllib.request.Request(PUBLIC_URL + "/v1/models", headers=h))))
batch = {"mod_name": "test", "entries": [
    {"id": 1, "key": "item.test.sword", "source": "Diamond Sword", "target_locale": "ja_jp"},
    {"id": 2, "key": "msg.test", "source": "%s killed %s with §cfire", "target_locale": "ja_jp"}]}
print(post({"model": MODEL_NAME, "messages": [{"role": "user", "content": json.dumps(batch)}]}))

In [ ]:
#@title ⑤ 記録・ログを見る
import json, os
N = 10  #@param {type:"integer"}
#@markdown 全件は左のファイル欄の `requests.jsonl` を右クリック → ダウンロード
if os.path.exists("/content/requests.jsonl"):
    for l in open("/content/requests.jsonl", encoding="utf-8").read().splitlines()[-N:]:
        e = json.loads(l)
        print("=" * 60)
        print(e["time"], e["status"])
        for m in (e["request"] or {}).get("messages", []):
            print(f"  [{m.get('role')}] {m.get('content')}")
        try:
            print("  [reply]", e["response"]["choices"][0]["message"]["content"])
        except Exception:
            print("  [reply]", e["response"])
else:
    print("まだ記録はありません")
print("\n--- proxy.log（末尾）---")
print(open("/content/proxy.log", encoding="utf-8").read()[-3000:])